### Required Background: Data-Dependent Acquisition (DDA) vs. Data-Independent Acquisition (DIA)
<img src='DDA vs. DIA.png' width=500>

**DDA**: Each fragmentation spectrum corresponds to a single peptide.

At every MS1 scan, the instrument selects the top k most intense precursor ions. Only these precursors are isolated, fragmented, and recorded as MS2 spectra This produces clean MS2 spectra, each representing one peptide. But the data is incomplete becuase of the precursor selection. 

**DIA**: A single peptide is reconstructed from multiple time-adjacent spectra collected across wide, overlapping m/z windows.

Instead of selecting specific precursors, DIA fragments every ion within a set of predefined m/z windows. These windows often overlap to avoid losing peptides at boundaries. A single peptide typically contributes fragments to many consecutive MS2 spectra.

This produces more complex MS2 spectra, but the data is certainly more complete because _all_ the precursors are fragmented. For a better visual of what I mean here, see the first few seconds of the video below.

---


## 4. Cascadia: Positional Embedding of Augmented Spectra

### 4.1 The Problem: What if one spectrum isn't enough?

In the previous notebook, we saw how Casanovo predicts peptide sequences over DDA data, whereas Cascadia predicts peptide sequences over DIA** data. So what happens when:
- Your peptide's signal is spread across multiple MS2 cycles (like in DIA)?
- or the MS1 isotope pattern holds crucial information about charge state and mass?
- or neighboring spectra contain corroborating evidence that could improve predictions?

Our motivation here is to leverage all contextual information available to make a better prediction. With this in mind, we can better understand the reasoning behind the "Augmented Spectra" concept introduced in Cascadia.

### Constructing an Augmented Spectrum

The following animation walks through the three step process for building an augmented spectrum from MS1 and MS2 data:

<video controls>
  <source src="AugmentedManim.mp4" type="video/mp4">
</video>

We perform the following:

(1) select a central MS2 scan

(2) include neighboring MS2 scans within a chosen width and,

(3) identify the corresponding MS1 scans in order to build the "augmented spectrum"

### 4.2 Building an Augmented Spectrum: Video => Data Structure


Here's how to read the augmented spectrum:

M/Z: The mass-to-charge ratio of a given peak

Normalized Intensity: Peak intensity, normalized to [0, 1]

Width: Relative position: `0` = central scan, `1` = one scan forward, `-1` = one scan backward

Scan Level: `2` = MS2 (fragment ions), `1` = MS1 (precursor/isotope peaks)

A full augmented spectrum is thus a collection of peaks represented as 4-tuples: (m/z, intensity, width, level).

---

#### Example Augmented Spectrum - Numeric

In our video, we used width = 2, meaning we selected 2 MS2 scans on each side of the central scan. However, the example below from [Cascadia's documentation](https://cascadia.readthedocs.io/en/latest/file_formats.html) uses width = 1** (1 scan on each side), which makes it easier to trace through.


Note that traditionally you might see a list of peaks from a single MS1 or MS2 scan listed as an array. This just "blows out" that array and lists each peak and intensity individually for some scan. 

```
BEGIN IONS
TITLE=1
PEPMASS=402.5
CHARGE=1
SEQ=PEPTIDEK
```

Central MS2 scan (Width = 0)
| M/Z | Intensity | Width | Level |
|-----|-----------|-------|-------|
| 185.08 | 0.67 | 0 | 2 |
| 367.85 | 0.41 | 0 | 2 |
| 400.98 | 0.88 | 0 | 2 |
| ... | ... | 0 | 2 |

Next MS2 scan (Width = +1)
| M/Z | Intensity | Width | Level |
|-----|-----------|-------|-------|
| 185.08 | 0.67 | 1 | 2 |
| 367.85 | 0.41 | 1 | 2 |
| ... | ... | 1 | 2 |

Previous MS2 scan (Width = -1)
| M/Z | Intensity | Width | Level |
|-----|-----------|-------|-------|
| 185.08 | 0.67 | -1 | 2 |
| 367.85 | 0.41 | -1 | 2 |
| ... | ... | -1 | 2 |

MS1 peaks within isolation window
| M/Z | Intensity | Width | Level |
|-----|-----------|-------|-------|
| 400.18 | 0.29 | 0 | 1 |
| 401.21 | 0.35 | 0 | 1 |
| 402.27 | 0.30 | -1 | 1 |
| ... | ... | ... | 1 |

```
END IONS
```

---

Once more, we must keep in mind our motivation. A single MS2 Spectrum in DIA is inherently ambigious. That is, it contains fragments from **all** precursors present in a given isolation window. By putting information across time (neighboring scans) + across MS levels (MS1 and MS2), this augmented spectra allows us to encode all of these details.


### 4.3 4-Tuple => Embedding

Each 4-tuple of  $\text{peak} = (\text{m/z}, \text{intensity}, \text{width}, \text{scan level})$ is transformed into a 512 dimensional embedding through these steps:

![AugmentedSpectrumArch.png](AugmentedSpectrumArch.png)

If you understood Casanovo well, Cascadia just adds a few more intricacies.

FOR ONE PEAK:

**1.** Gather its 4 raw values:
- m/z
- intensity
- width / scan offset / retention time
- level

Note that what we call scan offset or width, is actually retention time. It's an integer from $[-w,w]$ saying "this peak came from the central scan (0) or $w$ scans before or after." So what they label as "RT" in the figure is actually the **relative** retention time/position in the augmented window.

**2.** Peak encoder turns that 4-tuple into a single vector.

This is done in two parts:

(a) m/z is encoded using a sinusoidal positional embedding (like in Casanovo), AND so is width (retention time) in the exact same way. These two vectors are then concatenated. Call this vector $v_{\text{mz+rt}}$.

(b) Two values (intensity and level) are each passed through a learned linear layer to produce two vectors. 

What do I mean by "learned linear layer"? Before we answer that question, we'll start by defining our inputs and outputs. 

**Inputs:** Either intensity or level scalar values. Intensity is usually normalized to $[0, 1]$ beforehand, and level is either $0$ (MS1 scan) or $1$ (MS2 scan).

**Outputs:** We need to convert both intensity and level scalars into vectors $v_{\text{int}}$ and $v_{\text{level}}$ and concatenate them to $v_{\text{mz+rt}}$ such that we have just one vector $v_{\text{int+level+mz+rt}}$ that represents our intensity, level, m/z, and retention time all together. 

- Note that m/z and width are encoded using fixed sinusoidal embeddings, which don't change during training. This embedding simply maps values to vectors in such a way that preserves their relative positions.

Intensity and level, however, are not fixed at all. It depends on the actual data.

For example, a high intensity might mean a peak is more important, but the exact relationship depends on the relative strength to the dataset. To allow the model to ascertain the best way to use these values, we use a learned linear layer.

This is nothing more than a linear transformation. How we come up with that linear transformation is somewhat complicated though.


### Weights, Biases, and Backpropogation

We know now, that we need to come up with such a linear transformation that maps a number in dimension $1$ to a higher dimension to be able to concatenate all our data points together - something that looks like $y = mx + b$ but with vectors and matrices instead.

Here, I'll briefly mention the process to create this. For the sake of brevity, we'll just use intensity for now, but intensity and level are certainly swappable in the explanation.

The intensity scalar is passed through a learned linear layer:

$$
v_{\text{int}} = W_{\text{int}} \cdot \text{intensity} + b_{\text{int}}
$$

Where:

- $W_{\text{int}}$: A weight matrix that maps the scalar intensity to a vector of size $D_{\text{int}}$.
- $b_{\text{int}}$: A bias vector of size $D_{\text{int}}$.

At first, $W_{\text{int}}$ and  $b_{\text{int}}$ are initialized randomly. 

You might be wondering “How can the model know what to do with intensity alone?”

Nothing. This is the heart of representation learning.

It is only in Step 3, that we iteratively adjust $W_{\text{int}}$ and $b_{\text{int}}$ in order to minimize a chosen loss function (which'll likely be something to do with peptides correclty predicted)

This is a process known as backpropogation. We iterate backwards from the last layer (implemented in step 3) to the first, optimizing weights along the way. This is because Step 3 "knows" everything about our entire peak, whereeas right now, we are only given some intensity information.

This results in $W$ and $b$ that don't "work" immediately, but are later trained on and optimized with algorithms like SGD (stochastic gradient descent).

The important thing currently though, is that we have our representation of $v_{\text{int}} = W_{\text{int}} \cdot \text{intensity} + b_{\text{int}}$ and each intensity for each peak can be mapped to its vector representation with this linear transformation.

Accuracy is ensured later.

* Note that this holds without loss of generality to $v_{\text{level}}$ as well.



At last, we've constructed our $v_{\text{int+level+mz+rt}}$, with level and intensity acting as "placeholders" to be optimized.

**3.** Transformer + Linear + Softmax

Now that we have our peak embeddings $v_{\text{int+level+mz+rt}}$, we need to actually make those randomly initialized weights and biases *useful*. Remember, right now $W_{\text{int}}$, $W_{\text{level}}$, $b_{\text{int}}$, $b_{\text{level}}$​ are just random numbers.

This is where backpropagation comes in.

All peak embeddings are fed into a transformer encoder, which uses self-attention to capture relationships between peaks (like complementary ion pairs or isotope patterns). The transformer then outputs a contextualized spectrum representation.

This representation feeds into a transformer decoder that autoregressively predicts the peptide sequence one amino acid at a time. At each step, we calculate a "loss function" - essentially measuring how wrong our prediction was compared to the ground truth peptide.

Here's the key: we backpropagate this loss backwards through the entire network. From the decoder, through the transformer encoder, all the way back to those linear layers in Step 2. Using gradient descent, we iteratively adjust every weight and bias to minimize the loss.

Additionally, Cascadia includes an auxiliary fragment ion prediction task: each peak embedding passes through another linear layer to predict whether it's a b-ion, y-ion, or precursor. This adds extra training signal, helping the model learn which peaks are informative versus noise.

Training continues until the model converges - that is, until the loss stops decreasing.

#### Overall Architectural Details (move this to 03_Casanovo)

1. **Encoder:** Processes all peak embeddings together using self-attention. The model learns which peaks are related — e.g., a y-ion series, or matching peaks across consecutive scans.

2. **Decoder:** Generates amino acids autoregressively, one at a time. At each step, it attends to both the encoded spectrum and the previously generated amino acids.

3. **Beam Search with Mass Constraint:** Rather than greedily picking the most likely amino acid, Cascadia maintains multiple candidate sequences and prunes those whose cumulative mass exceeds the precursor mass.



### 4.5 Broad Overview - Why Augmented Spectra Work

We've gotten really into the weeds, but the overall idea is that this 4-tuple embedding allows the transformer to learn the about the following relationships:

- Peaks at the same m/z across consecutive scans -> likely the same fragment ion
- MS1 isotope spacing of 0.5 Da -> charge state = 2
- Peak intensity decreasing over time -> elution profile falling edge

TL;DR - Casanovo looks at one spectrum and predicts a sequence. Cascadia looks at a *neighborhood* of spectra and predicts a sequence, with more confidence and accuracy, especially for the challenging DIA case.